# QFF 2026 Phase 2 — Quantum Simulation starter notebook

Shows the **official measurement workflow** with the kit's tools. It does not solve any problem statement.
Run from `starter/` in the dedicated environment (`pip install -r ../requirements.txt`).

In [ ]:
import json, numpy as np
from scipy.optimize import minimize
import qiskit, qiskit_aer
from qff_sim import *
print('qiskit', qiskit.__version__, '| aer', qiskit_aer.__version__)
D = load_processor('processor_D_template.json')
print('Processor D check:', validate_processor_d(D, 4) or 'OK')

## 1. PS-S1 style: load an operator, exact references, a short HVA run

In [ ]:
d = json.load(open('../data/s1/tfim_open_N4_g1.00.json'))
H = SparsePauliOp.from_list([(l, c) for l, c in d['operator']])
print(d['meta']['boundary'], '| E0 =', d['meta']['E0_exact'], '| E1 =', d['meta']['E1_exact'])
qc, params = hva_ansatz(4, layers=2)
cost = CountedCost(lambda x: energy_statevector(qc.assign_parameters(dict(zip(params, x))), H))
res = minimize(cost, np.random.default_rng(0).uniform(-np.pi, np.pi, len(params)), method='COBYLA', options={'maxiter': 200})
bound = qc.assign_parameters(dict(zip(params, res.x)))
print(f"E = {res.fun:.5f}, error = {res.fun - d['meta']['E0_exact']:.5f}, calls = {cost.calls}")
ops = observable_ops(4)
print('observables (ideal):', {k: round(v, 4) for k, v in expectations(Statevector(bound), ops).items()})
print('observables (noisy on D):', {k: round(v, 4) for k, v in noisy_expectations(bound, ops, D).items()})

## 2. PS-S2 style: Trotter circuit vs exact evolution, CX vs RZZ construction

In [ ]:
H2 = tfim_hamiltonian(4, 1.0, 1.0)                    # open chain
psi0 = Statevector(trotter_circuit(4, 1.0, 1.0, 0.0, 0))  # Neel state '1010'
exact = exact_evolution(H2, psi0, [1.0])[0]
for n in (1, 4):
    st = Statevector(trotter_circuit(4, 1.0, 1.0, 1.0, n, order=1))
    print(f'n={n}: state error {state_error(st, exact):.4f}')
D_rzz = dict(D, basis_gates=['rz', 'sx', 'x', 'rzz'])
for mode in ('cx', 'rzz'):
    c = trotter_circuit(4, 1.0, 1.0, 1.0, 1, zz_mode=mode)
    print(mode, 'two-qubit gates per step on D(cx):', circuit_metrics(c, D)['cx'], '| on D(rzz):', circuit_metrics(c, D_rzz)['cx'])

## 3. PS-S3 style: density-matrix fidelity and Z-basis measurement on the ring

In [ ]:
Hr = tfim_hamiltonian(4, 1.0, 1.0, periodic=True)
target = exact_evolution(Hr, psi0, [2.0])[0]
c = trotter_circuit(4, 1.0, 1.0, 2.0, 6, periodic=True)
for P in (PROCESSOR_A, D):
    m = circuit_metrics(c, P)
    print(P['name'], '| 2q:', m['cx'], 'swaps:', m['swaps'], 'routing overhead:', m['cx_overhead'],
          '| F(noise x1):', round(noisy_state_fidelity(c, target, P), 4), 'F(noise x2):', round(noisy_state_fidelity(c, target, P, scale=2.0), 4))
print('Z-basis observables, 4096 shots, noisy on D:', run_z_measurement(c, D))

## 4. Objective 1 — Bell state on Processor D

In [ ]:
for pair in [(0, 1), (0, 8)]:
    for noisy in (False, True):
        r = bell_experiment(D, pair, noisy)
        print(pair, 'noisy' if noisy else 'ideal', f"F = {r['fidelity']:.4f} ± {r['fidelity_se']:.4f}")